# 1 环境依赖



# 2 数据结构

使用Nimbus输出的csv作为输入进行计算分析

# 3 处理逻辑

为每个marker计算对应的Otsu阈值，然后根据计算的阈值*用户指定的factor系数，确定每个细胞的每一个marker的状态（positive/negative）

# 4 原始代码

仅计算Otsu的阈值，对Nimbus的输出数据不做arcsinh转换拉伸，之前做arcsinh转换拉伸是为了在密度分布图视觉上让阴性峰和阳性峰区分更加显著。现在存代码的部分，对于计算机来说，arcsinh转换并没有任何帮助。另外从数据调度来说，原始数据和arcsinh转换后的数据进行Otsu的阈值计算时，对于我们的数据而言差异不大。所以本质上，当初进行arcsinh转换只是在作图时从视觉上让阴性峰和阳性峰的区分更加明显一些。所以在现在文章的数据中，并没有使用arcsinh转换，而是直接计算了Otsu阈值

In [ ]:
# ================= 1. 计算全样本全局 Otsu 阈值 (严谨版) =================
print("\n[Step 1] 计算每个样本的全局 Otsu 阈值 (基于 CSV 所有细胞)...")
SAMPLE_GLOBAL_OTSU = {}

markers = ['CD3', 'CD4', 'CD8a', 'TCR', 'EOMES']
key_map = {
    'CD3': 'Prob_CD3', 'CD4': 'Prob_CD4', 'CD8a': 'Prob_CD8',
    'TCR': 'Prob_TCR', 'EOMES': 'Prob_EOMES'
}

for sample in SAMPLES:
    nimbus_file = os.path.join(ROOT_DIR, sample, "nimbus_output", "nimbus_cell_table.csv")
    if not os.path.exists(nimbus_file):
        print(f"  [Warning] 找不到 CSV: {sample}")
        continue
    try:
        use_cols = [COL_MAP[key_map[m]] for m in markers]
        df_raw = pd.read_csv(nimbus_file, usecols=use_cols)
        SAMPLE_GLOBAL_OTSU[sample] = {}
        for m in markers:
            col_name = COL_MAP[key_map[m]]
            vals = df_raw[col_name].dropna().values
            if len(vals) > 0:
                try: otsu_val = filters.threshold_otsu(vals)
                except: otsu_val = np.mean(vals)
            else: otsu_val = 0.5
            SAMPLE_GLOBAL_OTSU[sample][m] = otsu_val
    except Exception as e:
        print(f"  [Error] 读取 {sample} 失败: {e}")

print("全局 Otsu 阈值计算完成。")

根据用户给定的系数，对marker的状态进行设置，随后输出一张新的csv表供用户在Qupath中重新导入，此处给出的是之前R的代码（保留了arcsinh转换），因为在最新的分析中已经完全脱离了Qupath，直接在python代码中确定了细胞类型。此处展示的代码只供逻辑理解

In [ ]:
# 1. 最佳系数配置 (Grid Search Top 1)
best_factors <- c(
  "CD3"     = 1.0,   
  "TCRbeta" = 0.6,   
  "EOMES"   = 1.6,   
  "CD4"     = 0.4,   
  "CD8a"    = 0.5,
  "CD45"    = 1.0
)

In [ ]:
mutate(
    # --- A. 判定原始阳性 (逻辑值) ---
    # 逻辑: Arcsinh值 > (Base_Otsu * Factor)
    is_Pos_CD3_Raw = CD3_Arcsinh > (Base_Thr_CD3 * best_factors["CD3"]),
    is_Pos_TCR_Raw = TCRbeta_Arcsinh > (Base_Thr_TCRbeta * best_factors["TCRbeta"]),
    is_Pos_EOM_Raw = EOMES_Arcsinh > (Base_Thr_EOMES * best_factors["EOMES"]),
    is_Pos_CD4_Raw = CD4_Arcsinh > (Base_Thr_CD4 * best_factors["CD4"]),
    is_Pos_CD8_Raw = CD8a_Arcsinh > (Base_Thr_CD8a * best_factors["CD8a"]),
    is_Pos_CD45_Raw = CD45_Arcsinh > (Base_Thr_CD45 * best_factors["CD45"]),
    
    # --- B. 互斥修正 (Mutex Correction) ---
    is_DP = is_Pos_CD4_Raw & is_Pos_CD8_Raw,
    
    # 修正后的逻辑状态
    # 如果是双阳，且 CD4 < CD8，则 CD4 判为 FALSE
    is_Pos_CD4_Final = ifelse(is_DP & (CD4_Arcsinh < CD8a_Arcsinh), FALSE, is_Pos_CD4_Raw),
    # 如果是双阳，且 CD8 < CD4，则 CD8 判为 FALSE
    is_Pos_CD8_Final = ifelse(is_DP & (CD8a_Arcsinh < CD4_Arcsinh), FALSE, is_Pos_CD8_Raw),
)

In [ ]:
mutate(
    Status_CD3 = ifelse(is_Pos_CD3_Raw, "Positive", "Negative"),
    Status_TCRbeta = ifelse(is_Pos_TCR_Raw, "Positive", "Negative"),
    Status_EOMES = ifelse(is_Pos_EOM_Raw, "Positive", "Negative"),
    Status_CD45 = ifelse(is_Pos_CD45_Raw, "Positive", "Negative"),
    
    # 注意：CD4/CD8 使用修正后的状态
    Status_CD4 = ifelse(is_Pos_CD4_Final, "Positive", "Negative"),
    Status_CD8 = ifelse(is_Pos_CD8_Final, "Positive", "Negative"),
    
    # 标记冲突修正记录 (可选，方便你检查哪些细胞被修正了)
    Mutex_Correction = ifelse(is_DP, "Corrected_DP", "None"),
    
    # 最终分类标签
    Cell_Type = case_when(
      Status_CD3 == "Positive" & Status_TCRbeta == "Negative" ~ "gd+T",
      
      Status_CD3 == "Positive" & Status_TCRbeta == "Positive" & Status_CD4 == "Positive" ~ "ab+CD4+T",
      
      Status_CD3 == "Positive" & Status_TCRbeta == "Positive" & Status_CD8 == "Positive" ~ "ab+CD8+T",
      
      Status_CD3 == "Negative" & Status_CD45 == "Positive" ~ "CD45+CD3-",
      
      Status_CD3 == "Positive" & Status_TCRbeta == "Positive" & Status_CD4 == "Negative" & Status_CD8 == "Negative" & Status_EOMES == "Negative"~ "ab+EOMES-DNT",
      
      Status_CD3 == "Positive" & Status_TCRbeta == "Positive" & Status_CD4 == "Negative" & Status_CD8 == "Negative" & Status_EOMES == "Positive"~ "ab+EOMES+DNT",
      
      Status_CD45 == "Positive" ~ "CD45+ Cells",
    )
  )